# Video 3: Tasks in CrewAI

## What We Will Build
In this notebook, we will build a **multi-step task workflow** that demonstrates:
- What tasks are and how they work
- Task description and expected output design
- Assigning agents to tasks
- Task context and dependencies
- Structured outputs
- Executing a complete multi-step workflow

By the end, you will have a research -> analysis -> writing pipeline with dependent tasks.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom langchain_openai import ChatOpenAIimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: Tasks in CrewAI

### What is a Task?
A **Task** is a specific assignment given to an agent. It defines:
- **What** needs to be done (description)
- **What good looks like** (expected output)
- **Who** does it (agent)

### Task Attributes
| Attribute | Description |
|-----------|-------------|
| `description` | Detailed instructions for the agent |
| `expected_output` | Clear definition of the deliverable |
| `agent` | The agent assigned to this task |
| `context` | Other tasks whose output this task depends on |
| `output_json` | Expected structured output format |
| `output_file` | File path to save the output |

### Task Dependencies
When Task B has Task A in its `context`, Task A must complete before Task B starts. The output of Task A is automatically passed to Task B.

## 4. Create Specialized Agents

In [ ]:
# Create agentsresearcher = Agent(    role='Senior Research Analyst',    goal='Gather comprehensive and accurate information on any topic',    backstory='An expert researcher with deep knowledge across multiple domains.',    llm=llm,    verbose=True,    allow_delegation=False)analyst = Agent(    role='Strategic Analyst',    goal='Transform raw data into actionable strategic insights',    backstory='A seasoned analyst who identifies patterns and trends others miss.',    llm=llm,    verbose=True,    allow_delegation=False)writer = Agent(    role='Professional Writer',    goal='Create clear, engaging, and well-structured content',    backstory='An award-winning writer who makes complex topics accessible.',    llm=llm,    verbose=True,    allow_delegation=False)print('Agents created successfully')

## 5. Create Research Task

The first task gathers raw information. No context needed since it's the starting point.

In [ ]:
# Task 1: Researchresearch_task = Task(    description=(        'Research the impact of artificial intelligence on the job market in 2025. '        'Cover: 1) Jobs being automated, 2) New jobs being created, 3) Skills in demand, '        '4) Industry-specific impacts. Provide specific data points and examples.'    ),    expected_output=(        'A comprehensive research report with specific data points, examples, and sources. '        'Minimum 400 words with clear sections.'    ),    agent=researcher)print('Research Task created')print(f'  Agent: {research_task.agent.role}')print(f'  Has Context: {research_task.context is not None}')

## 6. Create Analysis Task with Context

The analysis task depends on the research task. We add it to `context` so the analyst receives the research output.

In [ ]:
# Task 2: Analysis (depends on Research)analysis_task = Task(    description=(        'Analyze the research findings on AI and the job market. '        'Identify: 1) Key trends and patterns, 2) Opportunities for workers, '        '3) Risks and challenges, 4) Strategic recommendations for individuals and companies.'    ),    expected_output=(        'A strategic analysis report with trends, opportunities, risks, and actionable recommendations. '        'Minimum 300 words.'    ),    agent=analyst,    context=[research_task]  # <-- This task depends on research_task)print('Analysis Task created with dependency on Research Task')

## 7. Create Writing Task with Context

The writing task depends on both research and analysis. It receives outputs from both.

In [ ]:
# Task 3: Writing (depends on Research and Analysis)writing_task = Task(    description=(        'Write a comprehensive blog post about AI and the job market based on the research '        'and analysis provided. The blog should be engaging, informative, and accessible '        'to a general audience. Include a compelling title, introduction, body sections, and conclusion.'    ),    expected_output=(        'A polished blog post of at least 500 words with a compelling title, '        'clear structure, and engaging narrative. Include key insights from the analysis.'    ),    agent=writer,    context=[research_task, analysis_task]  # <-- Depends on both)print('Writing Task created with dependencies on Research and Analysis')

## 8. Build and Execute the Multi-Step Workflow

In [ ]:
# Create crew with all tasksworkflow_crew = Crew(    agents=[researcher, analyst, writer],    tasks=[research_task, analysis_task, writing_task],    process=Process.sequential,    verbose=True,    memory=False)print('Multi-step workflow crew built!')print(f'Execution order: Research -> Analysis -> Writing')print('\n' + '=' * 60)print('KICKING OFF WORKFLOW')print('=' * 60)result = workflow_crew.kickoff()

## 9. Display Final Output

In [ ]:
print('\n' + '=' * 60)print('FINAL OUTPUT')print('=' * 60)print(result)print('=' * 60)

## 10. Inspect Individual Task Outputs

We can access the output of each individual task after execution.

In [ ]:
print('\nIndividual Task Outputs:')print('=' * 60)print('\n--- Research Task Output ---')print(research_task.output.raw if hasattr(research_task.output, 'raw') else 'Output not available')print('\n--- Analysis Task Output ---')print(analysis_task.output.raw if hasattr(analysis_task.output, 'raw') else 'Output not available')print('\n--- Writing Task Output ---')print(writing_task.output.raw if hasattr(writing_task.output, 'raw') else 'Output not available')

## 11. Task Design Best Practices

1. **Be specific** in descriptions — vague tasks produce vague outputs
2. **Define expected_output clearly** — this is the quality benchmark
3. **Use context for dependencies** — don't repeat information
4. **One agent per task** — keeps accountability clear
5. **Keep tasks focused** — one main objective per task

---

## ✅ Summary

You have successfully:
1. ✅ Created a research task with detailed description and expected output
2. ✅ Created an analysis task with context dependency on research
3. ✅ Created a writing task with context from both previous tasks
4. ✅ Built a sequential crew that executes tasks in dependency order
5. ✅ Executed the multi-step workflow
6. ✅ Inspected individual task outputs

In the next notebook, we will explore **Crews and Process Types** (Sequential vs Hierarchical).